# LLM Inference vLLM 7B

![chat Bubbles](https://cdn-icons-png.flaticon.com/512/2076/2076246.png) ![GPU Illustration](https://cdn-icons-png.flaticon.com/512/4854/4854226.png)

**vLLM** is a high-performance inference and serving engine for large language models, optimised for speed and scalability. It delivers efficient text generation through innovations such as **PagedAttention**,** continuous batching**, and support for **quantisation**.

This is a template demonstrates on how to run **7B-class models** (e.g. Mistral, Llama, Gemma) on Saturn Cloud.

On [Saturn Cloud](https://saturncloud.io), you can scale from a single NVIDIA GPU to multi-GPU clusters, enabling distributed inference for larger models or higher throughput workloads — all within a managed, GPU-ready environment.

## 1. Install dependencies


We install **vLLM** and **Transformers**. A recent NVIDIA CUDA runtime is recommended for best performance.

In [ ]:
!pip install -q jedi
!pip install -q vllm transformers
!pip install uv
!uv venv vllm-env -p 3.12
!source vllm-env/bin/activate && uv pip install vllm
!source vllm-env/bin/activate && pip install ipykernel
!python -m ipykernel install --user --name=vllm-env --display-name "vLLM Env"




## 2. Environment check

Verify the GPU is visible and print library versions. Confirm the environment is GPU-enabled.

In [ ]:
import torch, platform
import vllm, transformers

cuda_ok = torch.cuda.is_available()
print(f"✅ CUDA available: {cuda_ok}")
if cuda_ok:
    print("🧠 GPU:", torch.cuda.get_device_name(0))
print("🧩 torch:", torch.__version__)
print("🧩 vllm:", vllm.__version__)
print("🧩 transformers:", transformers.__version__)
print("🐍 python:", platform.python_version())

## 3. Select model and vLLM settings

Choose a **7B** model from Hugging Face. The defaults below work with common, openly available options. If a model is gated, select a different one.

In [ ]:
# 🔧 Model & runtime config (edit these as needed)
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # e.g., "meta-llama/Llama-2-7b-chat-hf", "google/gemma-7b"
DTYPE = "auto"                 # "auto", "float16", "bfloat16", "float32"
TENSOR_PARALLEL = 1            # single GPU = 1
GPU_MEMORY_UTIL = 0.90         # 0.6–0.95 depending on VRAM
MAX_MODEL_LEN = 8192           # context length (depends on model)

## 4. Basic model inference

Load the model with **vLLM** and generate text for one or more prompts using **SamplingParams** (temperature, top_p, max_tokens, etc.).

In [ ]:
from vllm import LLM, SamplingParams

print("⏳ Loading model (this may download weights on first run)...")
llm = LLM(
    model=MODEL_ID,
    dtype=DTYPE,
    tensor_parallel_size=TENSOR_PARALLEL,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
)
print("✅ Model loaded!")


## 5. Sample prompts

Use the customise Let's test the model using sample prompts.

In [ ]:
# Example prompts
prompts = [
    "You are a helpful assistant. Summarise why efficient attention helps LLM inference.",
    "List three creative uses of a 7B model for education.",
]

# Sampling parameters
sampling = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256,
)

# Generate
outputs = llm.generate(prompts, sampling)
for out in outputs:
    print("\n---")
    print("Prompt:", out.prompt)
    print("Completion:", out.outputs[0].text.strip())


## 6. User Custom Prompt Testing

You can enter your prompt to test the model's chat capabilities here.

In [ ]:
# Helper function for quick generation
def generate_text(prompt, temperature=0.7, top_p=0.9, max_tokens=256):
    params = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_tokens)
    result = llm.generate([prompt], params)[0].outputs[0].text
    return result.strip()

print("\nQuick test:")
new_Prompt = input("Enter a prompt: ")
print(generate_text(new_Prompt))


# print(generate_text("Explain what continuous batching means in vLLM."))

## 7. Conclusion

You have successfully deployed and run a 7B-class Large Language Model using vLLM on Saturn Cloud. This template demonstrates how to perform high-speed inference, interact with your model via prompts, and scale seamlessly across single or multiple GPUs.


By using [Saturn Cloud’s GPU infrastructure](https://saturncloud.io/docs/user-guide/how-to/resources/), you can easily extend this workflow for larger models, API serving, or integrated data science pipelines — all within a managed, scalable environment designed for production-grade AI workloads. Visit [saturn cloud](https://saturncloud.io/) to easily deploy this model.